In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
logger = get_logger()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DecimalType, IntegerType, DateType
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

In [0]:
configs = parse_config_from_json("/Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/config/configs.json")

In [0]:
product_schema = StructType([StructField('product_id', StringType(), False), StructField('product_name', StringType(), False), StructField('category', StringType(), False), StructField('sub_category', StringType(), False), StructField('state', StringType(), False), StructField('price_per_product', DoubleType(), False)])


customer_schema = StructType([StructField('customer_id', StringType(), False), StructField('customer_name', StringType(), False), StructField('email', StringType(), False), StructField('phone', StringType(), False), StructField('address', StringType(), False), StructField('segment', StringType(), False), StructField('country', StringType(), False), StructField('city', StringType(), False), StructField('state', StringType(), False), StructField('postal_code', StringType(), False), StructField('region', StringType(), False)])




In [0]:
def fill_defaults(df: DataFrame, default_values: dict) -> DataFrame:
    """Fill missing values in a DataFrame with default values."""
    if df is None:
        raise ValueError("Cannot fill defaults for a None DataFrame.")
    if not default_values:
        return df

    try:
        logger.info("Filling missing values with defaults: %s", default_values)
        return df.fillna(default_values)
    except Exception as e:
        logger.error("Failed to fill defaults: %s", e)
        raise RuntimeError(f"Failed to fill defaults: {e}") from e

In [0]:
def dedupe(df: DataFrame, subset: list = None) -> DataFrame:
    """ Drop duplicate rows from a DataFrame."""
    
    return df.dropDuplicates(subset) if subset else df.dropDuplicates()

In [0]:
def apply_schema(df: DataFrame, schema: StructType) -> DataFrame:
    """Apply a schema to a DataFrame."""
    if df is None:
        raise ValueError("Cannot apply schema to a None DataFrame.")
    if not schema:
        raise ValueError("Schema must be a non-empty StructType.")
    try:
        result = df.select(*schema.names)
        logger.info("Schema applied: %s", schema.names)
        return result
    except Exception as e:
        logger.error("Failed to apply schema: %s", e)
        raise RuntimeError(f"Failed to apply schema: {e}") from e

In [0]:
def dq_customers(df: DataFrame) -> DataFrame:
    """
    Data quality checks for customer records.
    Rules:
    - customer_id: format XX-NNNNN 
    - customer_name: "Firstname Lastname"
    - postal_code: numeric only 
    - country: alphabetic only 
    - phone: numbers and special chars only, 
    - state, region, city: alphabetic
    - email: valid format if present 
    """
    validated = df.filter(
        # customer_id: not null/blank, matches XX-NNNNN pattern
        F.col("customer_id").isNotNull() &
        (F.trim(F.col("customer_id")) != "") &
        F.col("customer_id").rlike(r"^[A-Z]{2}-\d{5}$") &

        # customer_name: not null/blank, first + last name with SINGLE spaces only
        # rejects multi-space names like "B         ecky Martin"
        F.col("customer_name").isNotNull() &
        (F.trim(F.col("customer_name")) != "") &
        F.col("customer_name").rlike(r"^[A-Za-z]+( [A-Za-z]+)+$") &

        # postal_code: not null/blank, numeric only
        F.col("postal_code").isNotNull() &
        (F.trim(F.col("postal_code")) != "") &
        F.col("postal_code").rlike(r"^\d+$") &

        # country: not null/blank, alphabetic (letters and spaces only)
        F.col("country").isNotNull() &
        (F.trim(F.col("country")) != "") &
        F.col("country").rlike(r"^[A-Za-z\s]+$") &

        # state: alphabetic if present, null/blank allowed
        (F.col("state").isNull() |
         (F.trim(F.col("state")) == "") |
         F.col("state").rlike(r"^[A-Za-z\s]+$")) &

        # region: alphabetic if present, null/blank allowed
        (F.col("region").isNull() |
         (F.trim(F.col("region")) == "") |
         F.col("region").rlike(r"^[A-Za-z\s]+$")) &

        # city: alphabetic if present, null/blank allowed
        (F.col("city").isNull() |
         (F.trim(F.col("city")) == "") |
         F.col("city").rlike(r"^[A-Za-z\s]+$")) &

        # email: valid format if present, null/blank allowed
        (F.col("email").isNull() |
         (F.trim(F.col("email")) == "") |
         F.col("email").rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"))
    )
    return validated

In [0]:
def dq_products(df: DataFrame) -> DataFrame:
    """
    Data quality checks for product records.
    Rules:
    - No column should be null or blank
    - product_id: format XXX-XX-NNNNNNNN
    - product_name: not null/blank
    - category: alphabetic only 
    - sub_category: alphabetic only 
    - state: alphabetic only 
    - price_per_product: decimal, non-negative
    """
    validated = df.filter(
        # product_id: not null/blank, matches XXX-XX-NNNNNNNN pattern
        F.col("product_id").isNotNull() &
        (F.trim(F.col("product_id")) != "") &
        F.col("product_id").rlike(r"^[A-Z]{3}-[A-Z]{2}-\d{8}$") &

        # product_name: not null/blank
        F.col("product_name").isNotNull() &
        (F.trim(F.col("product_name")) != "") &

        # category: not null/blank, alphabetic (letters and spaces only)
        F.col("category").isNotNull() &
        (F.trim(F.col("category")) != "") &
        F.col("category").rlike(r"^[A-Za-z\s]+$") &

        # sub_category: not null/blank, alphabetic (letters and spaces only)
        F.col("sub_category").isNotNull() &
        (F.trim(F.col("sub_category")) != "") &
        F.col("sub_category").rlike(r"^[A-Za-z\s]+$") &

        # state: not null/blank, alphabetic (letters and spaces only)
        F.col("state").isNotNull() &
        (F.trim(F.col("state")) != "") &
        F.col("state").rlike(r"^[A-Za-z\s]+$") &

        # price_per_product: not null, decimal non-negative
        F.col("price_per_product").isNotNull() &
        (F.col("price_per_product") >= 0)
    )
    return validated

In [0]:
def build_dims(df: DataFrame, module_name: str) -> DataFrame:
    """
    Build dim_customer from validated stg_customers:
    1. Apply defaults
    2. Validate records
    3. Deduplicate by customer_id
    4. Generate surrogate key (customer_key)
    """

    defaults = configs[module_name]['defaults']
    module = configs[module_name]['module']
    target_table = f"az_ci_de_dbs.ecom_dev.dim_{module}"

    # Apply defaults first
    df = fill_defaults(df, defaults)
    
    if module == "customer":
        # data quality checks
        df = dq_customers(df)
        df = dedupe(df, ["customer_id"])
        # Apply schema
        df = apply_schema(df, customer_schema)
        # Generate surrogate key using monotonically_increasing_id
        df = df.withColumn("customer_key", F.monotonically_increasing_id())
    else:
        # Cast price_per_product to double (source is string)
        df = df.withColumn("price_per_product", F.col("price_per_product").cast(DoubleType()))
        # data quality checks
        df = dq_products(df)
        # dedupe
        df = dedupe(df)
        # Apply schema
        df = apply_schema(df, product_schema)
        # Generate surrogate key using monotonically_increasing_id
        df = df.withColumn("product_key", F.monotonically_increasing_id())
    
    delta_writer(df, target_table)

In [0]:
def main_build_dims():
    try:
        products_df = spark.read.table("az_ci_de_dbs.ecom_dev.stg_products")
        customers_df = spark.read.table("az_ci_de_dbs.ecom_dev.stg_customers")

        build_dims(products_df, "Products")
        build_dims(customers_df, "Customers")
    except Exception as e:
        logger.info(f"Error while building dims: {e}")
        raise e